# 11 — Publish the trained router to Hugging Face Hub

This uploads the already-trained DistilBERT router checkpoint from Google Drive to a public Hugging Face model repository.

No GPU is required.

Before running:
1. Create/sign in to Hugging Face.
2. Create a token with **write** permission.
3. In Colab, open the **Secrets** panel (key icon) and add `HF_TOKEN`.
4. Enable notebook access to that secret.

Do not paste the token directly into notebook code or GitHub.

In [ ]:
!pip -q install huggingface-hub==1.31.0

In [ ]:
from pathlib import Path
from google.colab import drive, userdata
from huggingface_hub import HfApi

drive.mount("/content/drive")

MODEL_DIR = Path("/content/drive/MyDrive/distilbert_llm_router")

if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Checkpoint folder not found: {MODEL_DIR}")

required = ["config.json", "model.safetensors", "tokenizer.json", "tokenizer_config.json"]
missing = [name for name in required if not (MODEL_DIR / name).exists()]

if missing:
    raise FileNotFoundError("Missing checkpoint files: " + ", ".join(missing))

print("Checkpoint folder:", MODEL_DIR)
for p in sorted(MODEL_DIR.iterdir()):
    print(" -", p.name)

In [ ]:
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is missing. Add a Hugging Face write token to Colab Secrets using the key icon.")

api = HfApi(token=HF_TOKEN)
account = api.whoami()
username = account["name"]

print("Authenticated Hugging Face user:", username)

In [ ]:
REPO_NAME = "cost-aware-llm-router-distilbert"
REPO_ID = f"{username}/{REPO_NAME}"

repo_url = api.create_repo(repo_id=REPO_ID, repo_type="model", private=False, exist_ok=True)

print("Model repository:", REPO_ID)
print("URL:", repo_url)

In [ ]:
api.upload_folder(
    folder_path=str(MODEL_DIR),
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Upload trained DistilBERT router checkpoint",
)

print("Checkpoint upload complete.")

In [ ]:
MODEL_CARD = """---
base_model: distilbert/distilbert-base-uncased
library_name: transformers
tags:
- llm-routing
- distilbert
- regression
- cost-aware-routing
- routerbench
---

# Cost-Aware LLM Router — DistilBERT Checkpoint

This repository contains the trained DistilBERT checkpoint used in the Cost-Aware LLM Router college ML project.

## Task
Given a benchmark-formatted question, the model predicts 11 bounded performance scores, one for each historical candidate LLM in the routing pool. The outputs are regression scores, not calibrated probabilities.

## Base model
`distilbert-base-uncased`

## Training data
The training pipeline uses the public RouterBench 0-shot data and focuses on MMLU, ARC-Challenge, and GSM8K / grade-school-math.

## Frozen routing experiment
The original frozen experiment selected threshold 0.94 using validation data only. On the untouched test split, DistilBERT router mean performance was 0.760448 versus 0.768067 for GPT-4 fixed, with historical mean cost 0.002877 versus 0.003738.

## Multi-seed robustness
Across three independent Kaggle training seeds (13, 42, 77): test router performance 0.761428 ± 0.001359; historical test cost 0.002966 ± 0.000093; gap vs GPT-4 0.664 ± 0.136 percentage points; cost saving 20.65% ± 2.49%.

## Important limitations
- The candidate model pool and costs are historical RouterBench values.
- Free-form real-world prompts may be out of distribution.
- The model predicts routing scores; it does not generate answers.
- Paired-bootstrap uncertainty extended beyond the 1-percentage-point margin, so strict statistical non-inferiority at that margin was not established.
- A simple domain-aware baseline is very strong on this restricted three-domain subset.

## Attribution
RouterBench: Qitian Jason Hu et al., ROUTERBENCH: A Benchmark for Multi-LLM Routing System, arXiv:2403.12031 (2024).
DistilBERT: Victor Sanh et al., DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter, arXiv:1910.01108 (2019).
"""

card_path = Path("/content/router_model_card.md")
card_path.write_text(MODEL_CARD, encoding="utf-8")

api.upload_file(path_or_fileobj=str(card_path), path_in_repo="README.md", repo_id=REPO_ID, repo_type="model", commit_message="Add model card")
print("Model card uploaded.")

In [ ]:
files = api.list_repo_files(repo_id=REPO_ID, repo_type="model")
print("Repository files:")
for name in files:
    print(" -", name)
print("\nROUTER_MODEL_ID =", REPO_ID)